Setup — load DRF and Eu152 spectrum once, reused by all cells below

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.mlem import run_mlem

drf = pd.read_csv("data/response_matrix.csv", index_col=0)
drf_energy = drf.index.values
h = drf.values                                         # shape (6100, 599)
e_true = 20 + 10 * np.arange(h.shape[1])               # known grid: 20-6000 keV, 10 keV steps

eu152_data = np.loadtxt("data/LaBr3 spectrum 1kb/LaBr3_Eu152_5mC.txt")
energy, counts = eu152_data[:, 0], eu152_data[:, 1]
counts_file = counts
y = np.zeros(h.shape[0])
y[:len(counts_file)] = counts_file

ba133_data = np.loadtxt("data/LaBr3 spectrum 1kb/LaBr3_Ba133_5mC.txt", skiprows=1)
new_energy = ba133_data[:, 0]

print(f"Loaded DRF with shape {h.shape}")
print(f"Loaded Eu152 spectrum with {len(counts_file)} points; y padded to {len(y)} points")

Raw file inspection — confirms no header row

In [ ]:
with open("data/LaBr3 spectrum 1kb/LaBr3_Ba133_5mC.txt") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i >= 3:
            break

Alignment check — single file, then all 5 sources

In [ ]:
print(f"new data:  starts at {new_energy[0]}, step {new_energy[1]-new_energy[0]}")
print(f"DRF axis:  starts at {drf_energy[0]}, step {drf_energy[1]-drf_energy[0]}")
print(f"aligned: {np.isclose(new_energy[0], drf_energy[0])}")

In [ ]:
folder = Path("data/LaBr3 spectrum 1kb")
for f in sorted(folder.glob("*.txt")):
    data = np.loadtxt(f)          # no skiprows — line 0 is real data
    energy_source = data[:, 0]
    step = energy_source[1] - energy_source[0]
    aligned = np.isclose(energy_source[0], drf_energy[0]) and np.isclose(step, drf_energy[1]-drf_energy[0])
    print(f"{f.name}: starts at {energy_source[0]}, step {step}, aligned: {aligned}")

Background check — quiet region above highest Eu152 line (1408 keV)

In [ ]:
plt.figure(figsize=(10, 5))
plt.semilogy(energy, np.clip(counts, 1, None))  # log scale, clip zeros so log doesn't break
plt.xlabel("Energy (keV)")
plt.ylabel("Counts (log scale)")
plt.title("Eu152 - raw spectrum")
plt.savefig("results/eu152_raw_check.png")
print(f"Min nonzero count: {counts[counts>0].min()}")
print(f"Max count (peak):  {counts.max()}")
print(f"Count in a quiet region (e.g. 1900-2000 keV, above highest Eu152 line 1408 keV): {counts[(energy>=1900)&(energy<2000)].mean():.2f}")

First MLEM pass — unsmoothed, sanity check against known Eu152 lines (122, 245, 344, 411, 444, 563, 586, 688, 778, 867, 964, 1086, 1112, 1213, 1299, 1408 keV)

In [ ]:
print(f"y: {len(counts_file)} real points padded to {h.shape[0]} to match H")

result_raw = run_mlem(y, h, n_iter=300)

top_idx = np.sort(np.argsort(result_raw.x)[::-1][:15])
print("\nTop reconstructed peaks:")
for i in top_idx:
    print(f"  E_true = {e_true[i]:5.0f} keV -> x = {result_raw.x[i]:.2f}")

Smoothing attempt (smooth_every=10, final_smooth=True) — finding: over-smooths, peaks broaden and drop ~50%, worse than unsmoothed on some lines

In [ ]:
result_smooth = run_mlem(y, h, n_iter=300, smooth_every=10, final_smooth=True)

top_idx = np.sort(np.argsort(result_smooth.x)[::-1][:15])
print("Top reconstructed peaks (smoothed):")
for i in top_idx:
    print(f"  E_true = {e_true[i]:5.0f} keV -> x = {result_smooth.x[i]:.2f}")

Comparison plot — unsmoothed vs smoothed overlay with known lines marked

In [ ]:
known_lines = [122, 245, 344, 411, 444, 563, 586, 688, 778, 867, 964, 1086, 1112, 1213, 1299, 1408]

plt.figure(figsize=(12, 6))
plt.plot(e_true, result_raw.x, label="unsmoothed", alpha=0.7)
plt.plot(e_true, result_smooth.x, label="smoothed (every 10, final)", alpha=0.7)
for line in known_lines:
    plt.axvline(line, color="gray", linestyle=":", linewidth=0.5)
plt.xlim(0, 1500)
plt.xlabel("E_true (keV)")
plt.ylabel("Reconstructed x")
plt.legend()
plt.title("Eu152 MLEM: unsmoothed vs smoothed")
plt.savefig("results/eu152_mlem_comparison.png")
print("saved results/eu152_mlem_comparison.png")